# Regional VMD EOG removal — frontal-native-axis-events

Gated native validation of unscaled axis support.

## Scientific question, inputs and contracts

Requires completed monotonic detector parent and passing controlled unscaled event arms. Reuse the exact separate-axis detector, calibration trials, correction and K3/2000 with original publisher evaluation EOG. Preserve complete-trial outputs, resting preservation checks and failure counts. No native retuning or reconstruction SNR claim.

## Fresh source and environment
Resolve latest published code once and pin it. New temporary clones avoid stale Python imports; original inputs remain immutable.

In [ ]:
import os, sys, json, tempfile, subprocess
from pathlib import Path
IN_KAGGLE = Path('/kaggle/input').is_dir()
if not IN_KAGGLE:
    raise RuntimeError('This numerical campaign runs on Kaggle only')
TEMP_ROOT = Path(tempfile.mkdtemp(prefix='vmd-eog-'))
CHECKOUT = TEMP_ROOT / 'repository'
REPOSITORY_URL = 'https://github.com/saltypal/Lateral-eye-Artifact-removal.git'
BRANCH = 'research/vmd-eog-removal-notebooks'
RUN_SPEC = {'stage': 'frontal-native-axis-events', 'run_id': 'frontal-native-axis-events-manual', 'git_sha': '', 'campaign_id': 'vmd-eog-20261010', 'profile': 'pilot'}
OUT = Path('/kaggle/working/results') / RUN_SPEC['campaign_id'] / RUN_SPEC['run_id']
OUT.mkdir(parents=True,exist_ok=False)
(OUT/'run_spec.json').write_text(json.dumps(RUN_SPEC,indent=2))
(OUT/'experiment.json').write_text(json.dumps(RUN_SPEC.get('experiment',{}),indent=2))
(OUT/'execution_state.json').write_text(json.dumps({'stage':RUN_SPEC['stage'],'status':'bootstrap_running'}))
print('BOOTSTRAP_START', RUN_SPEC['run_id'], RUN_SPEC['git_sha'], flush=True)
def bootstrap_call(command, phase):
    print('BOOTSTRAP_PHASE', RUN_SPEC['run_id'], phase, flush=True)
    result = subprocess.run(command)
    if result.returncode:
        (OUT/'execution_state.json').write_text(json.dumps({'stage':RUN_SPEC['stage'],
            'status':'failed','failure_layer':'bootstrap','phase':phase,'exit_code':result.returncode}))
        raise RuntimeError('Bootstrap failed during '+phase+'; retained run identity and status in '+str(OUT))
# Manual launches resolve latest once; CLI launches already have its exact SHA.
if not RUN_SPEC['git_sha']:
    line = subprocess.check_output(['git','ls-remote',REPOSITORY_URL,'refs/heads/'+BRANCH], text=True).strip()
    RUN_SPEC['git_sha'] = line.split()[0]
bootstrap_call(['git','clone','--quiet','--branch',BRANCH,'--single-branch',REPOSITORY_URL,str(CHECKOUT)], 'clone')
bootstrap_call(['git','-C',str(CHECKOUT),'checkout','--quiet',RUN_SPEC['git_sha']], 'checkout')
ACTUAL_SHA = subprocess.check_output(['git','-C',str(CHECKOUT),'rev-parse','HEAD'],text=True).strip()
if ACTUAL_SHA != RUN_SPEC['git_sha']:
    raise RuntimeError('Source SHA mismatch')
ENV_ROOT = TEMP_ROOT / 'environment'
bootstrap_call([sys.executable,'-m','venv','--without-pip','--system-site-packages',str(ENV_ROOT)], 'environment')
ENV_PY = ENV_ROOT / 'bin/python'
bootstrap_call([sys.executable,'-m','pip','--python',str(ENV_PY),'install','--quiet','--progress-bar','off','-r',str(CHECKOUT/'requirements.txt')], 'dependencies')
PROCESS_ENV = os.environ.copy()
PROCESS_ENV['PYTHONPATH'] = str(CHECKOUT/'src')
PROCESS_ENV['PYTHONUNBUFFERED'] = '1'
PROCESS_ENV['OMP_NUM_THREADS'] = '2'
(OUT/'run_spec.json').write_text(json.dumps(RUN_SPEC,indent=2))
(OUT/'experiment.json').write_text(json.dumps(RUN_SPEC.get('experiment',{}),indent=2))
print('Exact Git SHA:', ACTUAL_SHA, 'Stage:', RUN_SPEC['stage'], 'Output:',OUT,flush=True)


## Inspect the pinned algorithm contract
The source modules remain inspectable in this checkout. Mode vectors, references and paired targets have distinct roles; the accompanying explanation defines their shapes and inference boundaries.

In [ ]:
CONFIGURATION = json.loads((CHECKOUT/RUN_SPEC.get('config_path','configs/campaign.json')).read_text())
print('Declared signal contract:', {key: CONFIGURATION[key] for key in ('fs','window','hop','seed')})
print('Preservation limits:', CONFIGURATION['preservation'])
if CONFIGURATION.get('schema_version') == 3:
    print('Modules: frontal_campaign.py -> frontal_methods.py -> frontal_replication.py/frontal_native.py')
    print((CHECKOUT/'docs/FRONTAL_PAPER_AUDIT.md').read_text())
    print('Scope: verified frontal only; AutoVMD stopped; posterior unchanged')
else:
    print('Algorithm explanation: docs/SCIENTIFIC_EXPLANATION.md in the pinned checkout')
    print('Modules: signal.py -> reference.py/frontal.py/posterior.py -> experiments.py -> review evidence')
print('Git source:', ACTUAL_SHA)


## Execute reusable modules
HEOG/VEOG are teacher/evaluation information. Clean targets cannot enter inference. Failed numerical or scientific gates stop dependent work.

In [ ]:
command = [str(ENV_PY),'-m','vmd_eog.runner','--stage',RUN_SPEC['stage'],
           '--input','/kaggle/input','--output',str(OUT),'--profile',RUN_SPEC['profile'],
           '--config',RUN_SPEC.get('config_path','configs/campaign.json'),
           '--experiment',str(OUT/'experiment.json')]
process = subprocess.run(command,cwd=CHECKOUT,env=PROCESS_ENV)
if process.returncode:
    raise RuntimeError('Stage failed; inspect saved execution_state.json and kernel logs')
print((OUT/'execution_state.json').read_text())


## Saved configurations and result tables
These are immutable outputs from this run. Readiness, teacher eligibility and final scientific qualification are distinct decisions.

In [ ]:
import csv
from itertools import islice
for name in ('vmd_search.csv','posterior_search.csv','review_methods.csv','condition_source_uncertainty.csv','paper_metric_fixture.csv','paper_paired_channels.csv','paper_native_channels.csv','paper_rest_bandpower_channels.csv','paper_paired_source_means.csv','paper_native_source_means.csv','paper_paired_permutation_tests.csv','paper_native_permutation_tests.csv','native_condition_channels.csv','native_condition_participants.csv','native_condition_permutations.csv','vmd_diagnosis_method_means.csv','vmd_diagnosis_clean_preservation.csv','vmd_mode_projection_diagnostics.csv','paper_plot_participant_conditions.csv','vmd_sobi_method_means.csv','vmd_sobi_clean_preservation.csv'):
    path = OUT/name
    if path.exists():
        print(name, '(first 8 saved rows; complete table remains in outputs)')
        with path.open() as handle:
            display(list(islice(csv.DictReader(handle),8)))
for name in ('posterior_context_search.csv','posterior_context_oof_scores.csv'):
    path = OUT/name
    if path.exists():
        print(name, '(first 8 rows; complete paper metrics and failure records saved)')
        with path.open() as handle:
            display(list(islice(csv.DictReader(handle),8)))
for name in ('frontal_search.csv','frontal_oof_means.csv','frontal_oof_recipients.csv',
             'native_frontal_preservation_records.csv','native_frontal_preservation_participants.csv',
             'frontal_mechanism_diagnostics.csv','frontal_oof_conditions.csv','frontal_clean_preservation.csv'):
    path = OUT/name
    if path.exists():
        print(name, '(first 12 rows; complete table saved)')
        with path.open() as handle:
            display(list(islice(csv.DictReader(handle),12)))
for name in ('selected_frontal.json','selected_posterior.json','classical_gate.json','paper_evaluation_protocol.json','paper_metric_fixture_summary.json'):
    if (OUT/name).exists():
        print(name, (OUT/name).read_text()[:8000])
if (OUT/'frontal_controlled_gates.json').exists():
    print('Controlled engineering gates:', (OUT/'frontal_controlled_gates.json').read_text())
if (OUT/'frontal_controlled_review.md').exists():
    from IPython.display import Markdown
    display(Markdown((OUT/'frontal_controlled_review.md').read_text()))
review = OUT/'APPROACH_REVIEW.md'
if review.exists():
    from IPython.display import Markdown
    display(Markdown(review.read_text()))


## Inspect evidence and persist outputs
Native real EEG has no clean-reference SNR. Pass-through failures remain counted. Parameters and physiological frequency bands are distinct.

In [ ]:
from IPython.display import display, Image
for figure in sorted(OUT.glob('*.png')):
    display(Image(filename=str(figure)))
for report in sorted(OUT.glob('*summary.json')):
    print(report.name, report.read_text()[:16000])
print('Persist this completed kernel version and retrieve outputs before launching a dependent stage.')
